# RetrieveChat based FinRobot-RAG

In this demo, we showcase the RAG usecase of our finrobot, which inherits from autogen's RetrieveChat implementation.


Instead of using `RetrieveUserProxyAgent` directly, we register the context retrieval as a function for our bots.
For detailed implementation, refer to [rag function](../finrobot/functional/rag.py) and [rag workflow](../finrobot/agents/workflow.py) of `SingleAssistantRAG` 

In [1]:
import autogen
from autogen.cache import Cache
print("Success!")

d:\PartnaStudio\sentinel\stack\FinRobot-IntentChain\sentiment\venv\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


Success!


In [2]:
import autogen
from finrobot.agents.workflow import SingleAssistantRAG

for openai configuration, rename OAI_CONFIG_LIST_sample to OAI_CONFIG_LIST and replace the api keys

In [6]:
import os
from dotenv import load_dotenv
load_dotenv("sentiment/.env.local")

hf_api_key = os.getenv("HUGGINGFACE_API_KEY", "").strip('"\' ')
hf_model_name = os.getenv("HUGGINGFACE_MODEL_NAME_FEATHERLESS", "curiousily/Llama-3-8B-Instruct-Finance-RAG").strip('"\' ')
hf_base_url = os.getenv("HUGGINGFACE_BASE_URL", "https://router.huggingface.co/v1").strip('"\' ')

print(f"HF Model Name: {hf_model_name}")
print(f"HF Base URL: {hf_base_url}")
print(f"HF API Key exists: {bool(hf_api_key)}")

config_list = [
    {
        "model": hf_model_name,
        "base_url": hf_base_url,
        "api_key": hf_api_key,
        "max_tokens": 2048
    }
]

HF Model Name: curiousily/Llama-3-8B-Instruct-Finance-RAG:featherless-ai
HF Base URL: https://router.huggingface.co/v1
HF API Key exists: True


In [7]:
# Read OpenAI API keys from a JSON file
llm_config = {
    "config_list": config_list,
    "timeout": 120,
    "temperature": 0,
}

From `finrobot.agents.workflow` we import the `SingleAssistantRAG`, which takes a `retrieve_config` as input.
For `docs_path`, we first put our generated pdf report from [this notebook](./agent_annual_report.ipynb). 

For more configuration, refer to [autogen's documentation](https://microsoft.github.io/autogen/docs/reference/agentchat/contrib/retrieve_user_proxy_agent)

Then, lets do a simple Q&A.

In [8]:
assitant = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "vector_db": None,  # Autogen has bug for this version
        "docs_path": [
            "../report/Microsoft_Annual_Report_2023.pdf",
        ],
        "chunk_token_size": 1000,
        "get_or_create": True,
        "collection_name": "msft_analysis",
        "must_break_at_empty_line": False,
    },
)
assitant.chat("How's msft's 2023 income? Provide with some analysis.")

User_Proxy (to Data_Analyst):

How's msft's 2023 income? Provide with some analysis.

--------------------------------------------------------------------------------
[autogen.oai.client: 06-15 15:02:32] {329} WARNING - Model curiousily/Llama-3-8B-Instruct-Finance-RAG is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Data_Analyst (to User_Proxy):

Microsoft's 2023 income is expected to be around $245 billion, with a net income of $72 billion. This is based on the company's quarterly earnings reports and analyst estimates.

Microsoft's revenue is expected to grow by 10% year-over-year, driven by strong demand for its cloud computing services, including Azure and Microsoft 365. The company's gaming division, Xbox, is also expected to continue to grow, driven by the popularity of its console and PC games.

Microsoft's net income is expected to increase by 15% year-over-year, driven by h

Here we come up with a more complex case, where we put the 10-k report of MSFT here.

Let' see how the agent work this out.

In [9]:
assitant = SingleAssistantRAG(
    "Data_Analyst",
    llm_config,
    human_input_mode="NEVER",
    retrieve_config={
        "task": "qa",
        "vector_db": None,  # Autogen has bug for this version
        "docs_path": [
            "../report/2023-07-27_10-K_msft-20230630.htm.pdf",
        ],
        "chunk_token_size": 2000,
        "collection_name": "msft_10k",
        "get_or_create": True,
        "must_break_at_empty_line": False,
    },
    rag_description="Retrieve content from MSFT's 2023 10-K report for detailed question answering.",
)
assitant.chat("How's msft's 2023 income? Provide with some analysis.")

User_Proxy (to Data_Analyst):

How's msft's 2023 income? Provide with some analysis.

--------------------------------------------------------------------------------
[autogen.oai.client: 06-15 15:04:29] {329} WARNING - Model curiousily/Llama-3-8B-Instruct-Finance-RAG is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
Data_Analyst (to User_Proxy):

Microsoft's 2023 income is expected to be around $240 billion, with a net income of $70 billion. The company's revenue is expected to grow by 10% year-over-year, driven by strong demand for its cloud computing services, including Azure and Office 365.

Microsoft's cloud computing business has been a significant driver of its revenue growth, with Azure revenue increasing by 40% year-over-year. The company's gaming business, including Xbox and PC gaming, is also expected to continue to grow, with revenue increasing by 10% year-over-year.

Mic